## Embedding Techniques 

for converting the Text into vectors

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()  # to load all the environment variables

True

In [3]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_AI_KEY")

In [4]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")
embeddings


OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001AEA957FD50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001AEABF9A310>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [5]:
'''Maine just test krne ke loye exceptional handling lga diya hai taki 
easily code run kr ske'''


from openai import RateLimitError

text="I am learning OpenAI embeddings with LangChain"
try:
    query_result = embeddings.embed_query(text)
except RateLimitError:
    query_result = []
    print("OpenAI API credits are exhausted. Add credits or use a funded API key.")
query_result

OpenAI API credits are exhausted. Add credits or use a funded API key.


[]

In [6]:
len(query_result)

0

In [7]:
# We can set our own Dimension for the embeddings. 
# The default is 1536 for the text-embedding-3-large model.

embeddings_1024=OpenAIEmbeddings(model="text-embedding-3-large", embedding_ctx_length=1024)

In [8]:
'''Maine just test krne ke loye exceptional handling lga diya hai taki 
easily code run kr ske'''


from openai import RateLimitError

text="I am learning OpenAI embeddings with LangChain"
try:
    query_result = embeddings.embed_query(text)
except RateLimitError:
    query_result = []
    print("OpenAI API credits are exhausted. Add credits or use a funded API key.")
query_result

OpenAI API credits are exhausted. Add credits or use a funded API key.


[]

In [9]:
len(query_result)

0

## Check it for speech.txt

In [10]:
# Text loader
from langchain_community.document_loaders import TextLoader
loader=TextLoader("speech.txt")
docs=loader.load()
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness 

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
final_documents=text_splitter.split_documents(docs)
final_documents

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'speech.txt'}, page_

## String the Embeded Text into the vector database ChromaDB

1. Store it into the chroma Db 
2. Then use it for retrieval based on similarity score calculation

In [18]:
## Vector Embedding and Vector StoreDB
from langchain_community.vectorstores import Chroma


#using exceptional handling to avoid OpenAI API credits exhaustion if exhausted, we can use local embeddings instead.

from openai import RateLimitError
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

try:
    db = Chroma.from_documents(final_documents, embeddings_1024)

except RateLimitError:
    print("OpenAI API credits are exhausted. Using local embeddings instead.")


    ## Vector Embedding and Vector StoreDB

    db = None

    try:
        db = Chroma.from_documents(final_documents, embeddings_1024)
    except RateLimitError:
        print("OpenAI API credits are exhausted. No documents were added to the database.")

    db
    if db is None:
        pass

db


OpenAI API credits are exhausted. Using local embeddings instead.
OpenAI API credits are exhausted. No documents were added to the database.


In [ ]:
# Using DB for querying and retrieving relevant documents

query="It will be all the easier for us to conduct ourselves as belligerents"

retrieved_results=db.similarity_search(query)
print(retrieved_results)